In [1]:
!git clone https://github.com/Priyansh-81/Architectural-Immunology.git
%cd Architectural-Immunology
!pip install -U transformers accelerate bitsandbytes sentencepiece datasets

fatal: destination path 'Architectural-Immunology' already exists and is not an empty directory.
/content/Architectural-Immunology
  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached huggingface_hub-1.21.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached transformers-5.12.1-py3-none-any.whl (11.2 MB)
Using cached accelerate-1.14.0-py3-none-any.whl (389 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)
Using cached huggingface_hub-1.21.0-py3-none-any.whl (721 kB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling

In [2]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


In [3]:
import os
from getpass import getpass

token = getpass("HF token: ")

if token:
    os.environ["HF_TOKEN"] = token

HF token: ··········


In [4]:
print("Setup Complete")

Setup Complete


In [5]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [7]:
small_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

small_tokenizer = AutoTokenizer.from_pretrained(small_model_name)

small_model = AutoModelForCausalLM.from_pretrained(
    small_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Small model loaded")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Small model loaded


In [8]:
print("GPU memory allocated:",
      torch.cuda.memory_allocated()/1e9,
      "GB")

GPU memory allocated: 0.988097536 GB


In [9]:
reasoning_model_name = "Qwen/Qwen2.5-3B-Instruct"

reasoning_tokenizer = AutoTokenizer.from_pretrained(reasoning_model_name)

reasoning_model = AutoModelForCausalLM.from_pretrained(
    reasoning_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Reasoning model loaded")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Reasoning model loaded


In [10]:
print(
    "GPU memory allocated:",
    torch.cuda.memory_allocated()/1e9,
    "GB"
)

GPU memory allocated: 7.159975936 GB


In [11]:
prompt = "Question: What is 2+2?\nAnswer:"

inputs = reasoning_tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = reasoning_model.generate(
    **inputs,
    max_new_tokens=30
)

print(
    reasoning_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

Question: What is 2+2?
Answer: 4

Question: What is the capital of Peru?
Answer: Lima

Question: Which planet is closest to the Sun?
Answer: Mercury


